<a href="https://colab.research.google.com/github/YSM-arch/huggingface_llmcourse_colab/blob/llm-course/chapter3/section4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 一个完成的训练过程

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [ ]:
!pip install datasets evaluate transformers[sentencepiece]
!pip install accelerate
# To run the training on TPU, you will need to uncomment the following line:
# !pip install cloud-tpu-client==0.10 torch==1.9.0 https://storage.googleapis.com/tpu-pytorch/wheels/torch_xla-1.9-cp37-cp37m-linux_x86_64.whl

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset("glue", "mrpc")
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)


def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)


tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
tokenized_datasets = tokenized_datasets.remove_columns(["sentence1", "sentence2", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")
tokenized_datasets["train"].column_names

In [ ]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=8, collate_fn=data_collator
)
eval_dataloader = DataLoader(
    tokenized_datasets["validation"], batch_size=8, collate_fn=data_collator
)

In [ ]:
for batch in train_dataloader:
    break
{k: v.shape for k, v in batch.items()}

In [ ]:
from torch.optim import AdamW
from accelerate import Accelerator
from transformers import AutoModelForSequenceClassification, get_scheduler

def training_function():
  accelerator = Accelerator()

  model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
  optimizer = AdamW(model.parameters(), lr=3e-5)

  device = accelerator.device
  model.to(device )
  train_dl, eval_dl, model, optimizer = accelerator.prepare(
      train_dataloader, eval_dataloader, model, optimizer
  )

  num_epochs = 3
  num_training_steps = num_epochs * len(train_dl)
  lr_scheduler = get_scheduler(
      "linear",
      optimizer=optimizer,
      num_warmup_steps=0,
      num_training_steps=num_training_steps,
  )

  progress_bar = tqdm(range(num_training_steps))
  #进行训练循环
  model.train()
  for epoch in range(num_epochs):
      for batch in train_dl:
          outputs = model(**batch)
          loss = outputs.loss
          accelerator.backward(loss)

          optimizer.step()
          lr_scheduler.step()
          optimizer.zero_grad()
          progress_bar.update(1)
  #进行评估
  import evaluate

  metric = evaluate.load("glue", "mrpc")
  model.eval()
  for batch in eval_dataloader:
      batch = {k: v.to(device) for k, v in batch.items()}
      with torch.no_grad():
          outputs = model(**batch)

      logits = outputs.logits
      predictions = torch.argmax(logits, dim=-1)
      metric.add_batch(predictions=predictions, references=batch["labels"])

  result=metric.compute()
  print(result)

In [ ]:
from accelerate import notebook_launcher
from tqdm import tqdm

notebook_launcher(training_function)



 45%|████▌     | 621/1377 [01:23<01:42,  7.41it/s]

 45%|████▌     | 622/1377 [01:23<01:38,  7.68it/s]

 45%|████▌     | 623/1377 [01:24<01:38,  7.69it/s]

 45%|████▌     | 624/1377 [01:24<01:41,  7.42it/s]

 45%|████▌     | 625/1377 [01:24<01:41,  7.42it/s]

 45%|████▌     | 626/1377 [01:24<01:43,  7.27it/s]

 46%|████▌     | 627/1377 [01:24<01:47,  6.98it/s]

 46%|████▌     | 628/1377 [01:24<01:48,  6.94it/s]

 46%|████▌     | 629/1377 [01:24<01:51,  6.71it/s]

 46%|████▌     | 630/1377 [01:25<01:45,  7.05it/s]

 46%|████▌     | 631/1377 [01:25<01:46,  6.98it/s]

 46%|████▌     | 632/1377 [01:25<01:41,  7.36it/s]

 46%|████▌     | 633/1377 [01:25<01:43,  7.16it/s]

 46%|████▌     | 634/1377 [01:25<01:46,  6.95it/s]

 46%|████▌     | 635/1377 [01:25<01:46,  6.94it/s]

 46%|████▌     | 636/1377 [01:25<01:55,  6.41it/s]

 46%|████▋     | 638/1377 [01:26<01:39,  7.43it/s]

 46%|████▋     | 639/1377 [01:26<01:40,  7.35it/s]

 46%|████▋     | 640/1377 [01:26<01:41,  7.26it/s]

 47%|████▋